# Binary Skin Cancer Classification with Fine-Tuning
## Neural Networks — Final Project

---

### Description

This project implements and compares three deep neural network architectures
using **fine-tuning** for binary classification of skin lesions
(malignant vs. benign), using the **ISIC 2024 SLICE-3D Permissive** dataset.

---

### Dataset

| Feature              | Detail                                           |
|----------------------|--------------------------------------------------|
| **Name**             | ISIC 2024 — SLICE-3D Permissive                  |
| **Source**           | International Skin Imaging Collaboration (ISIC)  |
| **Images**           | ~200,000 skin lesion crops                       |
| **Resolution**       | ~128×128 px (smartphone-like images)             |
| **Task**             | Binary classification (malignant vs. benign)     |
| **Class imbalance**  | ~99.8% benign / ~0.2% malignant                      |
| **License**          | CC-BY 4.0                                        |

---

### Models

| Model           | Type                | Backbone           |
|-----------------|---------------------|--------------------|
| ResNet-50       | Classic CNN         | ResNet             |
| EfficientNet-B3 | Optimized CNN       | EfficientNet       |
| ViT-B/16        | Vision Transformer  | ViT                |

---

### Development Environment

- **Framework:** PyTorch 2.7.1 + CUDA 11.8
- **GPU:** NVIDIA RTX 3070 Laptop (8GB VRAM)
- **Python:** 3.10

---

### Evaluation Metrics

- AUC-ROC (primary metric — ISIC standard)
- F1-Score
- Precision and Recall
- Confusion matrix

In [ ]:
### libraries userd for the project
import torch
import pandas as pd
import os
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

In [127]:
### Check if PyTorch can access the GPU
print(torch.cuda.is_available())      # True
print(torch.cuda.get_device_name(0))  # RTX 3070
print(torch.__version__)

True
NVIDIA GeForce RTX 3070 Laptop GPU
2.7.1+cu118


In [128]:
### Open the truth CSV file with pandas
df = pd.read_csv("ISIC_2024_Permissive_Training_GroundTruth.csv")
print(df.head())
print(df.shape)
df['malignant'].value_counts()

        isic_id  malignant
0  ISIC_0015670        0.0
1  ISIC_0015845        0.0
2  ISIC_0015864        0.0
3  ISIC_0015902        0.0
4  ISIC_0024200        0.0
(217477, 2)


malignant
0.0    217183
1.0       294
Name: count, dtype: int64

In [129]:
class ISICDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.dataframe = dataframe
        self.img_dir = img_dir 
        self.transform = transform
    def __len__(self):
        return len(self.dataframe)
    def __getitem__(self, idx):
        img_name = self.dataframe.iloc[idx,0] + ".jpg"
        label = self.dataframe.iloc[idx,1]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        if self.transform: 
            image = self.transform(image)

        return image, label

In [130]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [131]:
#New dataset with labels and transformations
dataset = ISICDataset(dataframe=df, img_dir = "ISIC_2024_Permissive_Training_Input/ISIC_2024_Permissive_Training_Input", transform=transform)
print(len(dataset))

217477


In [132]:
# DataLoader for batching and shuffling
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [133]:
# Test the dataloader
images, labels = next(iter(dataloader))
print(images.shape)  # Should be [32, 3, 224, 224]
print(labels.shape)  # Should be [32]

torch.Size([32, 3, 224, 224])
torch.Size([32])
